# RahSaaf: train the local vehicle model

Run this in **Google Colab** or **Kaggle** with a GPU. In Colab: *Runtime → Change runtime type → T4 GPU*.

You need `split.zip`: the `data/dataset/split` folder from your laptop, zipped (made by `python -m training.split_dataset`).

Run the cells from top to bottom.

In [ ]:
!pip -q install ultralytics openvino
import torch
print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Turn it on in Runtime > Change runtime type."

## 1. Upload the dataset
Pick `split.zip` when asked. (On Kaggle, add it as a dataset instead and set `DATA` below to its folder.)

In [ ]:
from google.colab import files
uploaded = files.upload()
!rm -rf split && unzip -q split.zip -d .
!ls split

In [ ]:
DATA = "split/dataset.yaml"
!cat $DATA

## 2. Train

Starts from the stock YOLO11n weights and adapts them to your pictures.

- `epochs=100` with `patience=20`: stops early if 20 rounds pass with no improvement.
- With about 2,000 pictures this takes roughly 30–60 minutes on a T4.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data=DATA,
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    project="runs",
    name="rahsaaf",
    exist_ok=True,
)

## 3. Measure on the test set
These pictures were never used in training. Copy this table into your report.

In [ ]:
best = YOLO("runs/rahsaaf/weights/best.pt")
metrics = best.val(data=DATA, split="test", imgsz=640)
print(f"Overall mAP@50: {metrics.box.map50:.3f}")

## 4. Export for the laptop and download

Exports the wide OpenVINO version the live detector uses, and zips it together with the original weights.

In [ ]:
best.export(format="openvino", imgsz=[256, 416])
!cd runs/rahsaaf/weights && zip -qr /content/rahsaaf_model.zip best.pt best_openvino_model
files.download("/content/rahsaaf_model.zip")

## 5. Back on the laptop

Unzip into the project's `models/` folder, then:

```
mv models/best.pt models/local.pt
mv models/best_openvino_model models/local_openvino_model
uv run python tools/evaluate.py models/local.pt
uv run python -m worker.run_live SOURCE --zones zones.json --weights models/local_openvino_model
```